# **Importing Libraries**

In [ ]:
# Standard Library Imports
import os            # OS-level operations (paths, directory handling)
import math          # Mathematical functions
import shutil        # High-level file operations (copying, moving, deleting)

# Data Handling & Processing
import numpy as np               # Numerical computations and array operations
import pandas as pd              # Data manipulation and analysis
import h5py                      # Handling HDF5 file formats

# Progress Visualization
from tqdm import tqdm            # Progress bars for loops

# Plotting & Visualization
import matplotlib.pyplot as plt  # Plotting and visualization tools


# PyTorch Machine Learning Stack
import torch                     # Core PyTorch library
import torch.nn as nn            # Neural network layers and utilities
import torch.optim as optim      # Optimization algorithms (SGD, Adam, etc.)
from torch.utils.data import (   # Dataset and DataLoader utilities
    DataLoader,
    Dataset
)

# **Defining the architecture of sparse autoencoder**

In [ ]:
"""
Sparse Autoencoder model.

Anthropic-style ReLU SAE with L1 sparsity penalty and unit-norm decoder
columns, following Bricken et al. (2023).
"""

class SparseAutoencoder(nn.Module):
    """ReLU sparse autoencoder for decomposing neural network activations.

    Args:
        d_input: Dimensionality of the input activations.
        d_hidden: Dictionary size (number of latent features).
    """

    def __init__(self, d_input: int, d_hidden: int):
        super().__init__()
        self.d_input = d_input
        self.d_hidden = d_hidden
        self.W_enc = nn.Linear(d_input, d_hidden, bias=True)
        self.W_dec = nn.Linear(d_hidden, d_input, bias=True)
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0
            )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input activations into sparse feature activations."""
        return torch.relu(self.W_enc(x - self.W_dec.bias))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode sparse features back to activation space."""
        return self.W_dec(z)

    def forward(self, x: torch.Tensor):
        """Full forward pass: encode then decode.

        Returns:
            x_hat: Reconstructed activations.
            z: Sparse feature activations.
        """
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

    def loss(self, x: torch.Tensor, lam: float):
        """Compute SAE loss = MSE + lambda * L1.

        Returns:
            total_loss, mse, l1, z
        """
        x_hat, z = self.forward(x)
        mse = (x - x_hat).pow(2).mean()
        l1 = z.abs().mean()
        return mse + lam * l1, mse, l1, z

    def normalize_decoder(self):
        """Constrain decoder column norms to unity (call after each step)."""
        with torch.no_grad():
            self.W_dec.weight.data = nn.functional.normalize(
                self.W_dec.weight.data, dim=0
            )

# **Defining custom dataset class for training on activations**

In [ ]:
# Dataset class
class ActivationDataset(Dataset):
    def __init__(self, activation_path):

        self.x = torch.load(activation_path).float()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]

# **Define the training function**

In [ ]:
# Training
def train_sae(
    activation_path,
    save_dir,
    d_hidden,
    lam=1,
    epochs=50,
    batch_size=512,
    lr=1e-3,
    device="cuda"
):

    # Make the directory to save the trained models
    os.makedirs(save_dir, exist_ok=True)

    # get the activation dataset
    dataset = ActivationDataset(activation_path)

    # Define the dataloader
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    # Finding the embedding dimension in input
    d_input = dataset[0].shape[0]

    # Displaying the input dimension and dictionary size
    print(f"Input dimension : {d_input}")
    print(f"Dictionary size : {d_hidden}")

    # Define the sparse autoencoder model
    sae = SparseAutoencoder(
        d_input=d_input,
        d_hidden=d_hidden
    ).to(device)

    # define the optimizer
    optimizer = torch.optim.Adam(
        sae.parameters(),
        lr=lr
    )

    # Initialize the best loss
    best_loss = 1e9

    # For each epoch
    for epoch in range(epochs):
        # Set the sae to train mode
        sae.train()
        # Initialize the running loss, mse and l1 sparsity penalty loss to zero
        running_loss = 0
        running_mse = 0
        running_l1 = 0
        # tqdm taskbar
        pbar = tqdm(loader)
        # for each sample
        for x in pbar:
            # move the sample to device
            x = x.to(device)
            # Forward pass through sae and get loss, mse, l1 and activations
            loss, mse, l1, z = sae.loss(
                x,
                lam=lam
            )
            # reset the optimizer
            optimizer.zero_grad()
            # backward pass
            loss.backward()
            # make the update
            optimizer.step()
            # normalize the decoder
            sae.normalize_decoder()
            # accumulate the losses
            running_loss += loss.item()
            running_mse += mse.item()
            running_l1 += l1.item()
            # display the progress
            pbar.set_description(
                f"Epoch {epoch+1}"
            )
            pbar.set_postfix(
                loss=loss.item(),
                mse=mse.item(),
                l1=l1.item()
            )

        # calculate the epoch loss
        epoch_loss = running_loss / len(loader)
        # Print the losses
        print(
            f"Epoch {epoch+1:03d} | "
            f"Loss={epoch_loss:.6f} | "
            f"MSE={running_mse/len(loader):.6f} | "
            f"L1={running_l1/len(loader):.6f}"
        )

        # Save the best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(
                {
                    "model": sae.state_dict(),
                    "input_dim": d_input,
                    "hidden_dim": d_hidden,
                    "lambda": lam
                },

                os.path.join(
                    save_dir,
                    "best_sae.pt"
                )
            )
    print("Training Finished.")

# **Train separate SAE for each layer's activations**

In [ ]:
# Train the sae on temporal layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/train/temporal.pt",
    "sae_temporal",
    d_hidden=512
)
# Train the sae on spatial layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/train/spatial.pt",
    "sae_spatial",
    d_hidden=512

)
# Train the sae on lstm layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/train/lstm.pt",
    "sae_lstm",
    d_hidden=1024

)
# Train the sae on pooled layer activations
train_sae(
    "/kaggle/input/notebooks/sumanpunshi123/extract-activations/activations/train/pooled.pt",
    "sae_pooled",
    d_hidden=1024

)